# 02 · RI / Density Fitting in DFT

这一节讲 RI，也叫 density fitting (DF)。

对纯泛函 DFT 来说，RI 通常近似的是 Coulomb $J$，不是 XC 泛函本身：

$$
F^{\text{RKS}} = h + J_{\text{RI}} + V_{\text{xc}}
$$

$V_{\text{xc}}$ 仍然来自上一节的格点数值积分。

---

## 1 · RI 在近似什么？

四中心双电子积分：

$$
(\mu\nu|\lambda\sigma)
$$

RI 用辅助基 $\{P\}$ 展开 AO pair density，近似为：

$$
(\mu\nu|\lambda\sigma)
\approx
\sum_{PQ} (\mu\nu|P)(P|Q)^{-1}(Q|\lambda\sigma)
$$

于是给定密度矩阵 $D$，Coulomb 矩阵可以写成：

$$
J_{\mu\nu}^{\text{RI}} = \sum_P (\mu\nu|P)c_P
$$

其中：

$$
\sum_Q (P|Q)c_Q = \sum_{\lambda\sigma}(P|\lambda\sigma)D_{\lambda\sigma}
$$

这是一种低秩分解思路：用三中心积分和辅助基 metric 替代完整四中心张量。

---

## 2 · 分子、辅助基和积分

In [ ]:
import numpy as np
from scipy.linalg import eigh, fractional_matrix_power
from pyscf import gto, dft, df

mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="sto-3g")
auxbasis = "weigend"
auxmol = df.addons.make_auxmol(mol, auxbasis)

nao = mol.nao_nr()
naux = auxmol.nao_nr()
nelec = mol.nelectron
nocc = nelec // 2

S = mol.intor("int1e_ovlp_sph")
T = mol.intor("int1e_kin_sph")
V = mol.intor("int1e_nuc_sph")
H_core = T + V
A = fractional_matrix_power(S, -0.5)
E_nuc = mol.energy_nuc()

# Three-center and two-center Coulomb integrals.
# int3c[mu,nu,P] = (mu nu | P)
# int2c[P,Q]     = (P | Q)
int3c = df.incore.aux_e2(mol, auxmol, intor="int3c2e")
int2c = auxmol.intor("int2c2e")

eri = mol.intor("int2e_sph")  # only for checking exact J once

print(f"nao = {nao}, naux = {naux}, nelec = {nelec}")
print(f"int3c shape = {int3c.shape}")
print(f"int2c shape = {int2c.shape}")

---

## 3 · 从密度矩阵构造 RI-J

步骤非常直接：

1. 把 AO pair density 投影到辅助基：$b_P = \sum_{\mu\nu}(P|\mu\nu)D_{\mu\nu}$
2. 解 metric 方程：$(P|Q)c_Q=b_P$
3. 回代得到 $J_{\mu\nu}^{\text{RI}} = (\mu\nu|P)c_P$

In [ ]:
def get_occ(mo_energy):
    idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    mo_occ[idx[:nocc]] = 2.0
    return mo_occ


def make_rdm1(mo_coeff, mo_occ):
    C_occ = mo_coeff[:, mo_occ > 0]
    occ = mo_occ[mo_occ > 0]
    return (C_occ * occ) @ C_occ.T


def get_j_exact(dm):
    return np.einsum("ijkl,kl->ij", eri, dm, optimize=True)


def get_j_ri(dm):
    b = np.einsum("mnP,mn->P", int3c, dm, optimize=True)
    c = np.linalg.solve(int2c, b)
    return np.einsum("mnP,P->mn", int3c, c, optimize=True)

# Check RI-J against exact J for one trial density.
mo_energy0, mo_coeff0 = eigh(H_core, S)
dm0 = make_rdm1(mo_coeff0, get_occ(mo_energy0))
J_exact = get_j_exact(dm0)
J_ri = get_j_ri(dm0)

print(f"max|J_RI - J_exact| = {np.max(np.abs(J_ri - J_exact)):.3e} Hartree")
print(f"||J_RI - J_exact|| = {np.linalg.norm(J_ri - J_exact):.3e} Hartree")

---

## 4 · RKS + RI-J + DIIS

和上一节唯一不同：`get_j(dm)` 换成 `get_j_ri(dm)`。XC 仍然由 `numint.nr_rks` 做格点积分。

In [ ]:
def get_err_vec(fock, dm):
    return A @ (fock @ dm @ S - S @ dm @ fock) @ A


def apply_diis(fock_list, err_vec_list):
    n = len(fock_list)
    B = np.empty((n + 1, n + 1))
    B[-1, :] = -1.0
    B[:, -1] = -1.0
    B[-1, -1] = 0.0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.einsum("ij,ij->", err_vec_list[i], err_vec_list[j], optimize=True)

    rhs = np.zeros(n + 1)
    rhs[-1] = -1.0
    coeff = np.linalg.solve(B, rhs)[:-1]
    return np.einsum("i,ijk->jk", coeff, np.array(fock_list), optimize=True)


def energy_rks_df(dm, J, exc):
    e1 = np.einsum("ij,ji->", H_core, dm, optimize=True)
    e_coul = 0.5 * np.einsum("ij,ji->", J, dm, optimize=True)
    return e1 + e_coul + exc + E_nuc


def run_rks_df(xc, grid_level=3, conv_tol=1e-10, max_cycle=80, diis_space=8):
    grids = dft.gen_grid.Grids(mol)
    grids.level = grid_level
    grids.build()
    ni = dft.numint.NumInt()

    mo_energy, mo_coeff = eigh(H_core, S)
    dm = make_rdm1(mo_coeff, get_occ(mo_energy))

    e_old = 0.0
    fock_list = []
    err_vec_list = []

    for cycle in range(max_cycle):
        J = get_j_ri(dm)
        nelec_grid, exc, vxc = ni.nr_rks(mol, grids, xc, dm)
        fock = H_core + J + vxc
        e_tot = energy_rks_df(dm, J, exc)

        fock_list.append(fock)
        err_vec_list.append(get_err_vec(fock, dm))
        if len(fock_list) > diis_space:
            fock_list.pop(0)
            err_vec_list.pop(0)

        fock_diis = apply_diis(fock_list, err_vec_list) if cycle >= 2 else fock
        mo_energy, mo_coeff = eigh(fock_diis, S)
        dm_new = make_rdm1(mo_coeff, get_occ(mo_energy))

        dE = abs(e_tot - e_old)
        dD = np.linalg.norm(dm_new - dm)
        if dE < conv_tol and dD < 1e-8:
            return {"e_tot": e_tot, "cycles": cycle + 1, "nelec_grid": nelec_grid, "dm": dm}

        dm = dm_new
        e_old = e_tot

    raise RuntimeError(f"RI-RKS did not converge for xc={xc}")

print("RI-RKS functions ready.")

---

## 5 · 与 PySCF density fitting 对照

这里有两个参照：

1. `PySCF density_fit()`：应该和我们的 RI-J 代码高度一致
2. conventional PySCF RKS：没有 RI 近似，因此能量会有一个小差别

这个差别不是 DFT 泛函误差，而是辅助基引入的 RI 误差。

In [ ]:
for xc in ["lda,vwn", "pbe,pbe"]:
    ours = run_rks_df(xc, grid_level=3)

    mf_df = dft.RKS(mol).density_fit(auxbasis=auxbasis)
    mf_df.xc = xc
    mf_df.grids.level = 3
    mf_df.conv_tol = 1e-10
    mf_df.verbose = 0
    mf_df.kernel()

    mf_exact = dft.RKS(mol)
    mf_exact.xc = xc
    mf_exact.grids.level = 3
    mf_exact.conv_tol = 1e-10
    mf_exact.verbose = 0
    mf_exact.kernel()

    print(f"xc = {xc}")
    print(f"  ours RI-RKS:       {ours['e_tot']:.10f} Hartree")
    print(f"  PySCF density_fit: {mf_df.e_tot:.10f} Hartree   diff = {ours['e_tot'] - mf_df.e_tot:.2e}")
    print(f"  PySCF conventional:{mf_exact.e_tot:.10f} Hartree   RI error = {ours['e_tot'] - mf_exact.e_tot:.2e}")

---

## 6 · 小结

| 概念 | 含义 |
|------|------|
| RI / DF | 用辅助基分解四中心库仑积分 |
| 三中心积分 | $(\mu\nu|P)$ |
| metric | $(P|Q)$ |
| 在 pure DFT 中的位置 | 通常近似 Coulomb $J$ |
| 不被 RI-J 近似的部分 | XC grid integration $E_{xc}$ 和 $V_{xc}$ |
| 误差来源 | 辅助基不完备 |

对 WFT 背景的同学来说，RI-DFT 的 RI 和 RI-MP2、RI-CCSD 的核心数学对象是同一个：AO pair density 在 auxiliary basis 中的 Coulomb-metric projection。